In [6]:
!nvidia-smi

Fri Sep 18 17:08:20 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   59C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [7]:
!pip install pypdf
!pip install -q transformers einops accelerate langchain bitsandbytes
!pip install sentence_transformers
!pip install llama-index llama-index-llms-huggingface

In [8]:
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader
from llama_index.llms.huggingface import HuggingFaceLLM
from llama_index.core.prompts.prompts import SimpleInputPrompt

In [9]:
!mkdir Data

mkdir: cannot create directory ‘Data’: File exists


In [10]:
document = SimpleDirectoryReader(input_files=['Open_ai.pdf']).load_data()

In [11]:
document

[Document(id_='bdc3c382-5631-4393-ab62-c0947ffed80b', embedding=None, metadata={'file_path': 'Open_ai.pdf', 'file_name': 'Open_ai.pdf', 'file_type': 'application/pdf', 'file_size': 541036, 'creation_date': '2026-09-18', 'last_modified_date': '2026-09-18'}, excluded_embed_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], excluded_llm_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], relationships={}, metadata_template='{key}: {value}', metadata_separator='\n', text_resource=MediaResource(embeddings=None, data=None, text='%PDF-1.5\n%\n232 0 obj\n<< /Linearized 1 /L 541036 /H [ 2042 246 ] /O 236 /E 91691 /N 12 /T 539373 >>\nendobj\n                                                                                                         \n233 0 obj\n<< /Type /XRef /Length 78 /Filter /FlateDecode /DecodeParms << /Columns 5 /Predictor 12 >> /W [ 1 3 1 ] /Index [ 

In [12]:
system_prompt = """You are a Q&A assistant. Your goal is to answer questions as
accurately as possible based on the instructions and context provided."""

# This will wrap the default prompts that are internal to llama-index
query_wrapper_prompt = SimpleInputPrompt("<|USER|>{query_str}<|ASSISTANT|>")



In [39]:
from huggingface_hub import notebook_login
notebook_login()

In [34]:
import torch
from transformers import BitsAndBytesConfig

# Configure 8-bit quantization with CPU offloading enabled
quantization_config = BitsAndBytesConfig(
    load_in_8bit=True,
    llm_int8_enable_fp32_cpu_offload=True
)

llm = HuggingFaceLLM(
    context_window=4096,
    max_new_tokens=256,
    generate_kwargs={"temperature": 0.0, "do_sample": False},
    tokenizer_name="mistralai/Mistral-7B-Instruct-v0.3",
    model_name="mistralai/Mistral-7B-Instruct-v0.3",
    device_map="auto",
    #tokenizer_kwargs={"max_length": 4096},
    model_kwargs={
        "dtype": torch.float16,
        "quantization_config": quantization_config
    },
    system_prompt=system_prompt,
    query_wrapper_prompt=query_wrapper_prompt
)

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

In [1]:
!pip install langchain-community langchain-huggingface

In [20]:
!pip install -q llama-index-embeddings-langchain
from langchain_community.embeddings.huggingface import HuggingFaceEmbeddings
from llama_index.core import Settings
from llama_index.embeddings.langchain import LangchainEmbedding

embed_model = LangchainEmbedding(
  HuggingFaceEmbeddings(model_name="sentence-transformers/all-mpnet-base-v2")
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [21]:
from llama_index.core import Settings

Settings.llm = llm
Settings.embed_model = embed_model

In [27]:
index = VectorStoreIndex.from_documents(document)

In [38]:
query_engine = index.as_query_engine()
response = query_engine.query("What is the two-stage training procedure proposed in the paper?")

In [43]:
print(response.response)

The two-stage training procedure proposed in the paper involves first training a language model on a large corpus of text, and then fine-tuning this model on a smaller dataset that is specific to the task at hand.
